# Segmentation d'images avec PyTorch Lightning

Détourer les forêts sur des photographies aériennes de l'IGN : c'est un problème de segmentation, où le modèle classe chaque pixel, forêt ou non-forêt. Ce notebook entraîne et compare trois modèles, présentés dans les slides « Segmentation et UNet » :

1. un UNet simple, entraîné depuis zéro ;
2. un encodeur EfficientNet-B7 pré-entraîné sur ImageNet, suivi d'un décodeur sans connexions de saut : ce qu'on perd sans elles ;
3. un UNet dont l'encodeur est ce même EfficientNet-B7, d'abord gelé, puis fine-tuné.

Il demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Le clonage des données (3,9 Gio à télécharger) prend une dizaine de minutes : lancez les quatre cellules suivantes dès le début.

In [ ]:
!pip install -q lightning torchmetrics timm tifffile segmentation-models-pytorch

In [ ]:
import random

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import segmentation_models_pytorch as smp
import timm
import torch
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from tifffile import imread
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.v2 import functional as transforms

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!git clone --depth 1 https://github.com/mlambda/ign-FNF-2009.git

## Chargement des données

Les données sont des photographies aériennes prises par l'IGN en 2009 en Lorraine : 20 zones de 5 km × 5 km. Chaque zone est découpée en 9 × 9 = 81 tuiles de 1024 × 1024 pixels (1 pixel = 50 cm sur le terrain : une tuile couvre 512 m × 512 m).

- **Entrée** (fichiers `*_irc.tif`) : une image en *infrarouge couleur* (IRC). Ses trois canaux sont le proche infrarouge, le rouge et le vert, affichés en rouge, vert et bleu. La végétation réfléchit fortement l'infrarouge : elle apparaît en rouge.
- **Cible** (fichiers `*_pfnf_nch.tif`) : la classe de chaque pixel, forêt ou non-forêt, codée en *one-hot* sur deux canaux. Le code en garde l'indice : 1 pour forêt, 0 pour non-forêt. Au sens de l'IGN, une forêt n'est pas un simple groupe d'arbres : c'est une surface de plus de 5 000 m², large d'au moins 20 m, dont la végétation couvre au moins 10 % du sol et qui n'est pas utilisée pour l'agriculture.

2 zones ont été mises de côté pour la validation, et 2 autres pour le test. Toutes les tuiles d'une zone vont dans le même ensemble : des tuiles voisines se ressemblent trop pour mesurer honnêtement la généralisation. Il reste 16 zones pour l'entraînement, soit 1 296 tuiles.

In [ ]:
df_train = pandas.read_csv("ign-FNF-2009/train-files.csv")
df_val = pandas.read_csv("ign-FNF-2009/val-files.csv")
df_test = pandas.read_csv("ign-FNF-2009/test-files.csv")

print("Coordonnées en entraînement")
print(df_train.coord.unique())

print("Coordonnées en val")
print(df_val.coord.unique())

print("Coordonnées en test")
print(df_test.coord.unique())

df_test

## Étude d'un exemple et de son annotation

In [ ]:
def read_observation(path: str) -> torch.Tensor:
  """Read a tile as a (channels, height, width) tensor."""
  return torch.from_numpy(imread(path)).permute(2, 0, 1)


def read_annotation(path: str) -> torch.Tensor:
  """Read a one-hot annotation as a (1, height, width) mask."""
  return torch.from_numpy(imread(path)).argmax(dim=-1)[None].float()


example_input = read_observation(df_train.Observations[0])
example_annotation = read_annotation(df_train.Annotations[0])

In [ ]:
def to_displayable(image: torch.Tensor) -> numpy.ndarray:
  """Turn a (channels, height, width) tensor into an array imshow can plot."""
  image = image.detach().cpu()
  if image.shape[0] == 3:
    # Image en infrarouge couleur, canaux en dernier pour imshow
    return image.permute(1, 2, 0).numpy()
  # Cible ou prédiction : la carte de la classe forêt
  return image[-1].numpy()


def display_sample(*images: torch.Tensor) -> None:
  """Show side-by-side an input image, the ground truth and the prediction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titles = ["Entrée", "Cible", "Prédiction"]
  for image, ax, title in zip(images, axes, titles):
    image = to_displayable(image)
    ax.set_title(title)
    if image.ndim == 3:
      ax.imshow(image)
    else:
      # Bornes fixes : forêt (1) en jaune, non-forêt (0) en violet, même quand
      # toute la tuile est d'une seule classe
      ax.imshow(image, vmin=0, vmax=1)
    ax.axis("off")
  plt.show()


display_sample(example_input, example_annotation)

## Options générales

In [ ]:
# Dimension des images originales
dim_original = 1024

# Dimension des images en entrée du réseau
dim_input = 512

# Nombre de canaux des images en entrée
num_channel = 3

# Nombre de processus utilisés pour charger les données
num_workers = 2

# Nombre d'epochs du UNet entraîné depuis zéro, des modèles à encodeur
# pré-entraîné (gelé, seul leur décodeur apprend : ils convergent plus vite), puis
# du fine-tuning
epochs = 15
pretrained_epochs = 5
fine_tuning_epochs = 3

## Augmentation des données

Avec 1 296 tuiles d'entraînement, le réseau reverrait vite les mêmes images : on les augmente par des transformations aléatoires, qui le rendent moins sensible aux rotations, au cadrage, etc. Implémentez la fonction `transform_data`, qui reçoit une image `X`, de forme (3, 1024, 1024), et sa cible `Y`, de forme (1, 1024, 1024) :

- si `eval` vaut `True` (validation et test), un recadrage centré de `X` et de `Y` à la taille `cropped_size` ;
- sinon (entraînement), un recadrage aléatoire à cette taille, puis une rotation d'un nombre aléatoire de quarts de tour et des miroirs aléatoires.

La cible doit subir exactement la même transformation que l'image, sinon le masque ne correspond plus aux pixels. Vérifiez-le sur l'exemple, en décommentant la dernière ligne de la cellule. Le module `torchvision.transforms.v2.functional` est importé sous le nom `transforms`.

**Documentation utile**

- [`transforms.crop`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.functional.crop.html)
- [`torch.rot90`](https://docs.pytorch.org/docs/stable/generated/torch.rot90.html)
- [`transforms.horizontal_flip`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.functional.horizontal_flip.html)
- [`transforms.vertical_flip`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.functional.vertical_flip.html)
- [`random.randint`](https://docs.python.org/3/library/random.html#random.randint)

In [ ]:
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  # Votre code ici
  return X, Y


# display_sample(*transform_data(example_input, example_annotation))

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `height, width = X.shape[-2:]` : les dimensions de l'image sont les deux dernières du tenseur ; les tuiles font 1024 × 1024 pixels, et `cropped_size` vaut (512, 512)
- `transforms.crop(X, top, left, cropped_height, cropped_width)` : le coin en haut à gauche du recadrage, puis sa taille
- Recadrage centré : `top = (height - cropped_height) // 2` ; aléatoire : `random.randint(0, height - cropped_height)`, bornes comprises
- `torch.rot90(X, k, dims=(-2, -1))` : `k` quarts de tour dans le plan de l'image, avec `k = random.randint(0, 3)`
- `transforms.horizontal_flip(X)` et `transforms.vertical_flip(X)`, chacun une fois sur deux : `if random.random() < 0.5:`
- Tirez chaque paramètre aléatoire une seule fois, puis appliquez-le à `X` et à `Y`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  height, width = X.shape[-2:]
  cropped_height, cropped_width = cropped_size
  if eval:
    top, left = ..., ...  # recadrage centré
  else:
    top = random.randint(0, ...)
    left = random.randint(0, ...)
  X = transforms.crop(X, top, left, cropped_height, cropped_width)
  Y = transforms.crop(Y, top, left, cropped_height, cropped_width)
  if not eval:
    k = random.randint(0, 3)  # nombre de quarts de tour
    X, Y = torch.rot90(X, k, dims=(-2, -1)), torch.rot90(Y, k, dims=(-2, -1))
    if random.random() < 0.5:
      X, Y = ..., ...  # miroir gauche-droite
    if random.random() < 0.5:
      X, Y = ..., ...  # miroir haut-bas
  return X, Y
```

### Solution

In [ ]:
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  height, width = X.shape[-2:]
  cropped_height, cropped_width = cropped_size
  if eval:
    # Recadrage centré
    top = (height - cropped_height) // 2
    left = (width - cropped_width) // 2
  else:
    # Recadrage aléatoire
    top = random.randint(0, height - cropped_height)
    left = random.randint(0, width - cropped_width)
  X = transforms.crop(X, top, left, cropped_height, cropped_width)
  Y = transforms.crop(Y, top, left, cropped_height, cropped_width)
  if not eval:
    # Rotation d'un nombre aléatoire de quarts de tour
    k = random.randint(0, 3)
    X, Y = torch.rot90(X, k, dims=(-2, -1)), torch.rot90(Y, k, dims=(-2, -1))
    # Miroirs aléatoires, la même transformation pour l'image et la cible
    if random.random() < 0.5:
      X, Y = transforms.horizontal_flip(X), transforms.horizontal_flip(Y)
    if random.random() < 0.5:
      X, Y = transforms.vertical_flip(X), transforms.vertical_flip(Y)
  return X, Y


for _ in range(3):
  display_sample(*transform_data(example_input, example_annotation))

## Création d'un `Dataset` PyTorch

PyTorch lit les données exemple par exemple, dans un `torch.utils.data.Dataset`. Écrivez la classe `SegmentationDataset`, qui en hérite :

- `__init__` garde les chemins des images et des cibles, le mode (`eval`) et la taille du recadrage ;
- `__len__` renvoie le nombre de tuiles ;
- `__getitem__(index)` lit la tuile numéro `index` et sa cible avec `read_observation` et `read_annotation`, les transforme avec `transform_data`, et renvoie deux tenseurs : l'image, de forme (3, 512, 512), aux pixels ramenés entre 0 et 1, et la cible, de forme (1, 512, 512), qui vaut 1 pour la forêt.

Un `DataLoader`, fourni plus bas, regroupe ensuite ces exemples en batchs, mélange les tuiles d'entraînement à chaque epoch et parallélise leur lecture.

**Documentation utile**

- [`torch.utils.data.Dataset`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)

In [ ]:
class SegmentationDataset(Dataset):
  """Load samples with appropriate transforms."""

  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the dataset."""
    # Votre code ici

  def __len__(self) -> int:
    """Compute the number of samples."""
    # Votre code ici

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate one sample of data."""
    # Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `df.Observations.to_numpy()` et `df.Annotations.to_numpy()` : les chemins des images et des cibles, numérotés à partir de 0, même quand `df` est un extrait d'un autre tableau
- `read_observation(chemin)` : l'image, un tenseur `uint8` de forme (3, 1024, 1024) ; `read_annotation(chemin)` : la cible, de forme (1, 1024, 1024)
- `transform_data(X, Y, self.eval, self.cropped_size)`
- `X.float() / 255` : les pixels entre 0 et 1, en `float32`
- `len(SegmentationDataset(df_train, eval=True))` doit valoir 1296, et `SegmentationDataset(df_train, eval=True)[0][0].shape`, `torch.Size([3, 512, 512])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class SegmentationDataset(Dataset):
  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    self.X_paths = df.Observations.to_numpy()
    self.Y_paths = ...
    self.eval = eval
    self.cropped_size = cropped_size

  def __len__(self) -> int:
    return ...

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    X, Y = transform_data(
      read_observation(self.X_paths[index]),
      ...,  # cible
      self.eval,
      self.cropped_size,
    )
    return ..., Y  # pixels entre 0 et 1
```

### Solution

In [ ]:
class SegmentationDataset(Dataset):
  """Load samples with appropriate transforms."""

  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the dataset."""
    self.X_paths = df.Observations.to_numpy()
    self.Y_paths = df.Annotations.to_numpy()
    assert self.X_paths.shape[0] == self.Y_paths.shape[0]
    self.eval = eval
    self.cropped_size = cropped_size

  def __len__(self) -> int:
    """Compute the number of samples."""
    return self.X_paths.shape[0]

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate one sample of data."""
    X, Y = transform_data(
      read_observation(self.X_paths[index]),
      read_annotation(self.Y_paths[index]),
      self.eval,
      self.cropped_size,
    )
    # Pixels ramenés entre 0 et 1
    return X.float() / 255, Y


dataset = SegmentationDataset(df_train, eval=True)
X, Y = dataset[0]
len(dataset), X.shape, Y.shape

## Création du modèle

Le modèle passe d'un batch d'images de forme (batch, 3, 512, 512) à une carte de même taille, de forme (batch, 1, 512, 512) : pour chaque pixel, un *logit*, qu'une sigmoïde transforme en probabilité d'être de la forêt. Écrivez `get_model`, qui construit un UNet comme celui des slides « Implémenter un UNet simple en PyTorch » et « Passe avant du UNet », mais plus profond :

- `steps` niveaux, avec `starting_depth` canaux au premier, puis deux fois plus à chaque niveau, sans dépasser 512 : 32, 64, 128, 256 et 512 par défaut ;
- à chaque niveau, un `conv_block` plus riche que celui des slides : deux convolutions 3 × 3, chacune suivie d'une normalisation par batch, d'une ReLU et d'un *dropout* de 10 % ;
- l'encodeur divise la résolution par deux d'un niveau au suivant, de 512 à 32 pixels de côté ; le décodeur la double à chaque niveau, et concatène la sortie de l'encodeur de même résolution (connexion de saut) ;
- en sortie, une convolution 1 × 1 à un seul canal, sans activation : la perte applique la sigmoïde pendant l'entraînement, et la prédiction l'applique explicitement.

Cette architecture vient de [l'article qui présente U-Net](https://arxiv.org/abs/1505.04597).

**Documentation utile**

- [`nn.BatchNorm2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)
- [`nn.Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html)
- [`nn.ModuleList`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)

In [ ]:
def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `depths = [min(512, starting_depth * 2**i) for i in range(steps)]` : le nombre de canaux de chaque niveau
- `nn.Conv2d(in_channels, out_channels, 3, padding="same")` : `padding="same"` garde la taille de l'image
- `nn.BatchNorm2d(out_channels)`, `nn.ReLU()` et `nn.Dropout(0.1)`, enchaînés par `nn.Sequential`
- `nn.MaxPool2d(2)` divise la résolution par deux, `nn.Upsample(scale_factor=2)` la double
- `nn.ModuleList([...])` range les blocs de l'encodeur et ceux du décodeur : dans une simple liste Python, leurs poids seraient invisibles pour l'optimiseur
- `torch.cat([x, skip], dim=1)` : la connexion de saut, sur l'axe des canaux ; le bloc du décodeur reçoit donc les canaux des deux tenseurs
- `nn.Conv2d(depths[0], 1, 1)` : la couche de sortie
- `get_model()(torch.zeros(1, 3, 64, 64)).shape` doit valoir `torch.Size([1, 1, 64, 64])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
  modules = []
  for channels in [in_channels, out_channels]:
    modules += [
      nn.Conv2d(channels, out_channels, 3, padding="same"),
      ...,  # normalisation par batch
      ...,  # ReLU
      ...,  # dropout
    ]
  return nn.Sequential(*modules)


class UNet(nn.Module):
  def __init__(self, starting_depth: int = 32, steps: int = 5) -> None:
    super().__init__()
    depths = [min(512, starting_depth * 2**i) for i in range(steps)]
    # Encodeur : de num_channel à depths[0] canaux, puis d'un niveau au suivant
    self.encoder = nn.ModuleList(
      [conv_block(i, o) for i, o in zip([num_channel, *depths], depths)]
    )
    # Décodeur : du fond du « U » vers le haut, chaque bloc reçoit le niveau
    # inférieur (below canaux) et la connexion de saut (depth canaux)
    self.decoder = nn.ModuleList(
      [conv_block(..., depth) for below, depth in zip(depths[:0:-1], depths[-2::-1])]
    )
    self.downsample = nn.MaxPool2d(2)
    self.upsample = ...
    self.head = ...

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    skips = []
    for i, block in enumerate(self.encoder):
      x = block(self.downsample(x) if i else x)
      skips.append(x)
    for block, skip in zip(self.decoder, reversed(skips[:-1])):
      x = block(torch.cat([..., skip], dim=1))  # résolution doublée
    return ...


def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  return UNet(starting_depth, steps)
```

### Solution

In [ ]:
# Le modèle U-Net de l'article https://arxiv.org/abs/1505.04597 (voir aussi
# les slides sur le UNet)


def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
  """Two 3 × 3 convolutions, each with batch normalization, ReLU and dropout."""
  modules = []
  for channels in [in_channels, out_channels]:
    modules += [
      nn.Conv2d(channels, out_channels, 3, padding="same"),
      nn.BatchNorm2d(out_channels),
      nn.ReLU(),
      nn.Dropout(0.1),
    ]
  return nn.Sequential(*modules)


class UNet(nn.Module):
  """U-Net: an encoder, a decoder, and skip connections between the two."""

  def __init__(self, starting_depth: int = 32, steps: int = 5) -> None:
    super().__init__()
    depths = [min(512, starting_depth * 2**i) for i in range(steps)]
    # Encodeur : un bloc par niveau, de num_channel à depths[0] canaux, puis d'un
    # niveau au suivant
    self.encoder = nn.ModuleList(
      [conv_block(i, o) for i, o in zip([num_channel, *depths], depths)]
    )
    # Décodeur : du fond du « U » vers le haut, chaque bloc reçoit la sortie du
    # niveau inférieur (below canaux), suréchantillonnée, concaténée à celle de
    # l'encodeur de même résolution (depth canaux)
    self.decoder = nn.ModuleList(
      [
        conv_block(below + depth, depth)
        for below, depth in zip(depths[:0:-1], depths[-2::-1])
      ]
    )
    self.downsample = nn.MaxPool2d(2)
    self.upsample = nn.Upsample(scale_factor=2)
    # Un logit par pixel : la sigmoïde est appliquée par la perte
    self.head = nn.Conv2d(depths[0], 1, 1)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # Encodeur : la résolution est divisée par deux d'un niveau au suivant, et la
    # sortie de chaque niveau est gardée pour la connexion de saut
    skips = []
    for i, block in enumerate(self.encoder):
      x = block(self.downsample(x) if i else x)
      skips.append(x)
    # Décodeur : la résolution est doublée, puis la sortie de l'encodeur de même
    # résolution est concaténée (le fond du « U », skips[-1], n'en a pas)
    for block, skip in zip(self.decoder, reversed(skips[:-1])):
      x = block(torch.cat([self.upsample(x), skip], dim=1))
    return self.head(x)


def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  return UNet(starting_depth, steps)


get_model()(torch.zeros(1, num_channel, 64, 64)).shape

## Entraînement

La cellule suivante fournit le `LightningModule` du notebook, `ImageSegmenter` : il décrit la perte, les métriques et l'optimiseur de n'importe quel modèle de segmentation binaire. Le modèle renvoie des logits : la perte `binary_cross_entropy_with_logits` leur applique la sigmoïde pendant l'entraînement, et `predict_step` l'applique explicitement.

On suit la justesse (*accuracy*), la part des pixels bien classés, et l'IoU de la classe forêt (slide « Pertes et métriques en segmentation ») : la surface où la prédiction et la cible disent toutes deux « forêt », divisée par la surface où au moins l'une des deux le dit. Elle vaut 1 pour un masque parfait, et 0 pour un modèle qui ne prédit jamais la forêt, quelle que soit sa justesse : c'est sur elle qu'on choisit le meilleur modèle.

In [ ]:
class ImageSegmenter(lightning.LightningModule):
  """Lightning wrapper for binary segmentation models."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-4) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, num_channel, dim_input, dim_input)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(task="binary")
        for stage in ("train", "val", "test")
      }
    )
    # IoU (indice de Jaccard) de la classe positive : contrairement à la justesse,
    # elle n'est pas gonflée par les pixels d'une classe majoritaire
    self.ious = nn.ModuleDict(
      {
        f"{stage}_iou": torchmetrics.classification.BinaryJaccardIndex()
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    images, annotations = batch
    logits = self(images)
    loss = nn.functional.binary_cross_entropy_with_logits(logits, annotations)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits.sigmoid(), annotations.int())
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    iou = self.ious[f"{stage}_iou"]
    iou(logits.sigmoid(), annotations.int())
    self.log(f"{stage}_iou", iou, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(
    self, batch: torch.Tensor | tuple[torch.Tensor, ...], batch_index: int
  ) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).sigmoid().cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

In [ ]:
batch_size = 4


def make_loader(df: pandas.DataFrame, batch_size: int, eval: bool) -> DataLoader:
  """Wrap a dataset in a data loader: batching, shuffling and prefetching."""
  return DataLoader(
    SegmentationDataset(df, eval),
    batch_size=batch_size,
    shuffle=not eval,
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
  )


# Chargeurs d'entraînement et de validation : `num_workers` processus lisent les
# batchs suivants pendant que le GPU calcule
train_loader = make_loader(df_train, batch_size, eval=False)
val_loader = make_loader(df_val, batch_size, eval=True)

# Scores de test de chaque modèle, comparés à la fin du notebook
test_results: dict[str, dict[str, float]] = {}

Le notebook entraîne quatre modèles de la même façon. Écrivez la fonction `train(segmenter, name, epochs, loader)`, sur le modèle des slides « Entraîner avec le `Trainer` » et « Par early stopping ». Elle crée un `Trainer` de Lightning qui :

- entraîne `segmenter` pendant `epochs` epochs sur `loader`, en l'évaluant sur `val_loader` à la fin de chaque epoch, sur le GPU s'il y en a un ;
- écrit les métriques de chaque epoch dans le dossier `logs/<name>` ;
- arrête l'entraînement quand la perte de validation n'a pas baissé depuis 5 epochs, et sauvegarde les poids de meilleure `val_iou`.

Elle renvoie ce `Trainer` : `trainer.test(..., ckpt_path="best")` rechargera ces meilleurs poids.

Entraînez ensuite le UNet de `get_model`, dans un `ImageSegmenter` de pas 1e-4, sous le nom `unet`. Lightning affiche la progression de chaque epoch.

**Documentation utile**

- [`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [`CSVLogger`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.loggers.csv_logs.html#lightning.pytorch.loggers.csv_logs.CSVLogger)
- [`EarlyStopping`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.EarlyStopping.html)
- [`ModelCheckpoint`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.ModelCheckpoint.html)

In [ ]:
def train(
  segmenter: ImageSegmenter,
  name: str,
  epochs: int = epochs,
  loader: DataLoader = train_loader,
) -> lightning.Trainer:
  # Votre code ici
  ...


# Votre code ici : le UNet de get_model, entraîné avec un pas de 1e-4

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `lightning.Trainer(max_epochs=epochs, accelerator="auto", devices=1, logger=..., callbacks=[...])` : `accelerator="auto"` choisit le GPU s'il y en a un
- `CSVLogger("logs", name=name)` : les métriques de chaque epoch, dans `logs/<name>/version_0/metrics.csv`
- `TQDMProgressBar()`, en tête des callbacks : la barre de progression de tqdm, plutôt que celle de rich, que Lightning choisit quand rich est installé (Colab) et qui a figé des entraînements
- `EarlyStopping(monitor="val_loss", patience=5)`
- `ModelCheckpoint(monitor="val_iou", mode="max", save_weights_only=True, save_top_k=1)` : `mode="max"`, puisqu'une IoU plus grande est meilleure ; `save_top_k=1` ne garde que le meilleur fichier
- `trainer.fit(segmenter, loader, val_loader)`
- `ModelSummary(segmenter, max_depth=2)` affiche les couches du modèle, la forme de leurs sorties et le nombre de paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def train(
  segmenter: ImageSegmenter,
  name: str,
  epochs: int = epochs,
  loader: DataLoader = train_loader,
) -> lightning.Trainer:
  trainer = lightning.Trainer(
    max_epochs=epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    callbacks=[
      TQDMProgressBar(),
      EarlyStopping(monitor=..., patience=5),
      ModelCheckpoint(monitor=..., mode=..., save_weights_only=True, save_top_k=1),
    ],
  )
  trainer.fit(..., loader, val_loader)
  return trainer


segmenter = ImageSegmenter(get_model(), learning_rate=...)
print(ModelSummary(segmenter, max_depth=2))
trainer = train(segmenter, "unet")
```

### Solution

In [ ]:
def train(
  segmenter: ImageSegmenter,
  name: str,
  epochs: int = epochs,
  loader: DataLoader = train_loader,
) -> lightning.Trainer:
  """Entraîne `segmenter` et garde les poids de meilleure IoU de validation."""
  trainer = lightning.Trainer(
    max_epochs=epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    callbacks=[
      # La barre de progression de tqdm, plutôt que celle de rich que Lightning
      # choisit quand rich est installé (Colab) : elle a figé des entraînements
      TQDMProgressBar(),
      EarlyStopping(monitor="val_loss", patience=5),
      # Les poids de meilleure IoU de validation, que `trainer.test` rechargera
      ModelCheckpoint(
        monitor="val_iou", mode="max", save_weights_only=True, save_top_k=1
      ),
    ],
  )
  # Évaluation sur le chargeur de validation à la fin de chaque epoch
  trainer.fit(segmenter, loader, val_loader)
  return trainer


segmenter = ImageSegmenter(get_model(), learning_rate=1e-4)
print(ModelSummary(segmenter, max_depth=2))
trainer = train(segmenter, "unet")

## Application aux données de test

La cellule suivante crée le chargeur de test, et place dans `X` et `Y` 8 tuiles de test qui ont des lisières : entre 20 et 80 % de forêt dans la partie recadrée que voit le modèle.

Évaluez le modèle sur tout le chargeur de test, avec les poids de sa meilleure epoch, et rangez ses scores dans `test_results["UNet entraîné depuis zéro"]`. Affichez ensuite ses prédictions pour les 8 tuiles de `X` avec `display_sample`.

Les prédictions sont-elles de bonne qualité ? Comparez l'IoU de test à celle d'un modèle qui ne prédirait jamais la forêt, et regardez où se trouvent les erreurs.

**Documentation utile**

- [`Trainer.test`](https://lightning.ai/docs/pytorch/stable/common/trainer.html#test)
- [`Trainer.predict`](https://lightning.ai/docs/pytorch/stable/common/trainer.html#predict)

In [ ]:
test_loader = make_loader(df_test, 8, eval=True)

# 8 tuiles de test qui ont des lisières : entre 20 et 80 % de forêt dans la partie
# recadrée que voit le modèle
crop = (dim_original - dim_input) // 2


def forest_share(path: str) -> float:
  forest = read_annotation(path)[0, crop : crop + dim_input, crop : crop + dim_input]
  return forest.mean().item()


shares = df_test.Annotations.map(forest_share)
df_edges = df_test[shares.between(0.2, 0.8)].sample(8, random_state=0)
edges_loader = DataLoader(SegmentationDataset(df_edges, eval=True), batch_size=8)
X, Y = next(iter(edges_loader))
X.shape, Y.shape

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")` : recharge dans `segmenter` les poids sauvegardés par le `ModelCheckpoint`, puis renvoie une liste d'un dictionnaire par chargeur, `[{"test_loss": …, "test_accuracy": …, "test_iou": …}]`
- `trainer.predict(segmenter, edges_loader)` : une liste de tenseurs de probabilités de forêt, un par batch ; `torch.cat(...)` les met bout à bout, en un tenseur de forme (8, 1, 512, 512)
- `for x, y, p in zip(X, Y, preds):` parcourt les images, les cibles et les prédictions ensemble

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
scores = trainer.test(segmenter, dataloaders=..., ckpt_path="best")[0]
test_results["UNet entraîné depuis zéro"] = scores
print(test_results)
preds = torch.cat(trainer.predict(segmenter, ...))
for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)
```

### Solution

In [ ]:
# `ckpt_path="best"` recharge les poids de meilleure IoU de validation
scores = trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")[0]
test_results["UNet entraîné depuis zéro"] = scores
print(test_results)

# Probabilités de forêt de chaque pixel des 8 tuiles : `predict` renvoie un tenseur
# par batch, mis bout à bout
preds = torch.cat(trainer.predict(segmenter, edges_loader))
for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)

Un modèle qui ne prédit jamais la forêt a une IoU nulle, même avec une bonne justesse : l'IoU de test du UNet en est loin, et les grandes zones de forêt et de non-forêt sont bien détourées. Les erreurs se concentrent :

- le long des lisières, où la prédiction hésite (probabilités intermédiaires) et où la cible elle-même n'est précise qu'à quelques pixels près ;
- sur les petits objets, haies, bosquets, clairières ou chemins forestiers, que la définition de l'IGN (plus de 5 000 m², plus de 20 m de large) ne classe pas toujours comme l'œil ;
- sur la végétation qui n'est pas de la forêt, prairies et cultures, rouge elle aussi en infrarouge couleur.

## Modèle pré-entraîné sans connexions de saut

Le UNet précédent apprend tout depuis zéro, encodeur compris, sur 1 296 tuiles. Un encodeur pré-entraîné sur ImageNet sait déjà extraire des bords, des textures et des formes : c'est l'apprentissage par transfert (slides « Transfer learning »). ImageNet ne contient pas d'images infrarouges, mais ces caractéristiques visuelles restent utiles.

La cellule suivante charge EfficientNet-B7 avec `timm`, sans sa tête de classification, et fournit deux modules :

- `Normalize` normalise les pixels, déjà ramenés entre 0 et 1, avec la moyenne et l'écart-type de chaque canal sur ImageNet : les entrées que le réseau a vues pendant son entraînement ;
- `Frozen` gèle un module, comme dans la slide « Geler le backbone » : ses poids ne bougent plus, et ses BatchNorm gardent les statistiques d'ImageNet.

Pour une tuile de 512 × 512, EfficientNet-B7 rend des cartes de 16 × 16 × 2560.

Écrivez un décodeur volontairement simple, **sans connexions de saut** : il ne fait que suréchantillonner la sortie de l'encodeur gelé, de 16 × 16 jusqu'à 512 × 512. Ce modèle montre ce qu'on perd sans elles : l'encodeur reconnaît la forêt, mais les détails fins calculés par ses premières couches n'atteignent jamais le décodeur.

Entraînez-le avec `train`, avec un pas de 1e-3 (seul le décodeur, parti de zéro, apprend) pendant `pretrained_epochs` epochs, sous le nom `no_skip`. Rangez ses scores de test dans `test_results["EfficientNet-B7 sans connexions de saut"]`, et comparez ses prédictions sur `X` à celles du UNet.

In [ ]:
class Normalize(nn.Module):
  """Normalize images already scaled to [0, 1], channel by channel."""

  def __init__(
    self, mean: tuple[float, ...] | float, std: tuple[float, ...] | float
  ) -> None:
    super().__init__()
    self.register_buffer("mean", torch.tensor(mean).reshape(1, -1, 1, 1))
    self.register_buffer("std", torch.tensor(std).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.mean) / self.std


class Frozen(nn.Module):
  """Wrap a module whose weights are frozen and kept in evaluation mode."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.requires_grad_(False)
    self.eval()

  def train(self, mode: bool = True) -> "Frozen":
    # BatchNorm et dropout restent en mode évaluation
    return super().train(False)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


# EfficientNet-B7 pré-entraîné sur ImageNet, sans sa tête de classification
base_model = timm.create_model(
  "tf_efficientnet_b7.aa_in1k", pretrained=True, num_classes=0, global_pool=""
)
# Moyenne et écart-type de chaque canal sur ImageNet
data_config = timm.data.resolve_model_data_config(base_model)
data_config["mean"], data_config["std"], base_model.num_features

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Sequential(Normalize(data_config["mean"], data_config["std"]), Frozen(base_model), ...)` : la normalisation, l'encodeur gelé, puis les couches du décodeur
- Une convolution 1 × 1 suivie d'une ReLU réduit d'abord les 2560 canaux à 512
- Cinq fois de suite : `nn.Upsample(scale_factor=2)`, qui double la résolution, puis une convolution 3 × 3 (`padding="same"`) suivie d'une ReLU, avec 256, 128, 64, 32 puis 16 canaux en sortie
- `nn.Dropout(0.1)`, puis `nn.Conv2d(16, 1, 3, padding="same")` : le logit de chaque pixel
- Puis les mêmes appels que pour le UNet : `ImageSegmenter(model, learning_rate=1e-3)`, `train`, `trainer.test`, `trainer.predict`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def conv(in_channels: int, out_channels: int, kernel_size: int = 3) -> nn.Sequential:
  return nn.Sequential(
    nn.Conv2d(in_channels, out_channels, kernel_size, padding="same"), nn.ReLU()
  )


decoder = [conv(base_model.num_features, 512, kernel_size=1)]  # 16 × 16 × 512
for in_channels, out_channels in zip([512, 256, 128, 64, 32], [256, 128, 64, 32, 16]):
  decoder += [..., conv(in_channels, out_channels)]  # résolution doublée
model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  Frozen(base_model),  # 16 × 16 × 2560
  *decoder,  # 512 × 512 × 16
  nn.Dropout(0.1),
  ...,  # un logit par pixel
)

segmenter = ImageSegmenter(model, learning_rate=1e-3)
print(ModelSummary(segmenter, max_depth=2))
trainer = train(segmenter, "no_skip", epochs=pretrained_epochs)
scores = ...
test_results["EfficientNet-B7 sans connexions de saut"] = scores
preds = torch.cat(trainer.predict(segmenter, edges_loader))
for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)
```

### Solution

In [ ]:
def conv(in_channels: int, out_channels: int, kernel_size: int = 3) -> nn.Sequential:
  """A convolution followed by a ReLU, keeping the image's size."""
  return nn.Sequential(
    nn.Conv2d(in_channels, out_channels, kernel_size, padding="same"), nn.ReLU()
  )


# Décodeur : de 16 × 16 à 512 × 512, en doublant 5 fois la résolution, sans rien
# recevoir des couches intermédiaires de l'encodeur
decoder = [conv(base_model.num_features, 512, kernel_size=1)]  # 16 × 16 × 512
for in_channels, out_channels in zip([512, 256, 128, 64, 32], [256, 128, 64, 32, 16]):
  decoder += [nn.Upsample(scale_factor=2), conv(in_channels, out_channels)]
model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  Frozen(base_model),  # 16 × 16 × 2560
  *decoder,  # 512 × 512 × 16
  nn.Dropout(0.1),
  nn.Conv2d(16, 1, 3, padding="same"),
)

# Seul le décodeur, parti de zéro, apprend : un pas plus grand que pour le UNet
segmenter = ImageSegmenter(model, learning_rate=1e-3)
print(ModelSummary(segmenter, max_depth=2))
trainer = train(segmenter, "no_skip", epochs=pretrained_epochs)
scores = trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")[0]
test_results["EfficientNet-B7 sans connexions de saut"] = scores
print(test_results)

# Les mêmes tuiles de test que pour le UNet
preds = torch.cat(trainer.predict(segmenter, edges_loader))
for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)

Comparez ses masques à ceux du UNet, et le détail des contours à celui de la cible : le décodeur reconstruit les contours à partir de cartes de 16 × 16 seulement, où un pixel couvre 32 × 32 pixels de la tuile. L'encodeur pré-entraîné reconnaît la forêt, mais sans connexions de saut, le décodeur sait moins précisément *où* passent ses bords : cela se voit sur les lisières, pas forcément dans l'IoU.

## UNet pré-entraîné

Combinons les deux : un vrai UNet, avec connexions de saut, dont l'encodeur est le même EfficientNet-B7 pré-entraîné (slide « UNet avec un encodeur pré-entraîné »). La bibliothèque `segmentation_models_pytorch` (smp) l'assemble : seules les connexions de saut le distinguent du modèle précédent.

1. Créez le UNet de smp, avec l'encodeur `tu-tf_efficientnet_b7` pré-entraîné sur ImageNet (le préfixe `tu-` désigne un modèle de `timm`) et une seule classe en sortie.
2. Gelez son encodeur avec `Frozen` : seul le décodeur apprend d'abord.
3. Placez `Normalize` devant, comme pour le modèle précédent.
4. Entraînez-le comme le précédent, sous le nom `pretrained_unet`, puis rangez ses scores de test dans `test_results["UNet pré-entraîné, encodeur gelé"]`, et comparez ses prédictions sur `X` à celles des deux modèles précédents.

**Documentation utile**

- [`smp.Unet`](https://smp.readthedocs.io/en/latest/models.html#unet)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `smp.Unet(encoder_name="tu-tf_efficientnet_b7", encoder_weights="imagenet", classes=1)` : un logit par pixel en sortie
- `unet.encoder = Frozen(unet.encoder)` : gèle l'encodeur, ses poids comme les statistiques de ses BatchNorm
- `nn.Sequential(Normalize(data_config["mean"], data_config["std"]), unet)` : les pixels normalisés comme ceux d'ImageNet, puis le UNet
- `ImageSegmenter(pretrained_unet, learning_rate=1e-3)`, puis `train(segmenter, "pretrained_unet", epochs=pretrained_epochs)` : seul le décodeur, parti de zéro, apprend
- `trainer.test(..., ckpt_path="best")` recharge aussi les meilleurs poids dans `pretrained_unet` : le fine-tuning repartira de là

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
unet = smp.Unet(encoder_name=..., encoder_weights="imagenet", classes=1)
unet.encoder = ...  # encodeur gelé
pretrained_unet = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]), unet
)

segmenter = ImageSegmenter(pretrained_unet, learning_rate=1e-3)
print(ModelSummary(segmenter, max_depth=2))
trainer = train(segmenter, "pretrained_unet", epochs=pretrained_epochs)
scores = ...
test_results["UNet pré-entraîné, encodeur gelé"] = scores
preds = ...
for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)
```

### Solution

In [ ]:
unet = smp.Unet(
  encoder_name="tu-tf_efficientnet_b7", encoder_weights="imagenet", classes=1
)
# Étape 1 : l'encodeur est gelé (poids et statistiques des BatchNorm), seul le
# décodeur apprend
unet.encoder = Frozen(unet.encoder)
pretrained_unet = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]), unet
)

segmenter = ImageSegmenter(pretrained_unet, learning_rate=1e-3)
print(ModelSummary(segmenter, max_depth=2))
trainer = train(segmenter, "pretrained_unet", epochs=pretrained_epochs)
# `ckpt_path="best"` recharge aussi les meilleurs poids dans `pretrained_unet`, d'où
# repartira le fine-tuning
scores = trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")[0]
test_results["UNet pré-entraîné, encodeur gelé"] = scores
print(test_results)

preds = torch.cat(trainer.predict(segmenter, edges_loader))
for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)

### Fine-tuning de l'encodeur

Une fois le décodeur entraîné, dégelez l'encodeur, et poursuivez l'apprentissage de tout le réseau pendant `fine_tuning_epochs` epochs (slide « Fine-tuning : dégeler le backbone »), avec AdamW et deux groupes de paramètres :

- un pas de 1e-5 pour l'encodeur, dont les poids sont déjà bons : un grand pas les détruirait ;
- un pas de 1e-4 pour le décodeur, déjà entraîné : 10 fois plus petit qu'à l'étape précédente.

Pour cela, écrivez `FineTuningSegmenter`, un `ImageSegmenter` dont `configure_optimizers` crée ces deux groupes. Tout le réseau étant entraîné, il faut plus de mémoire GPU : utilisez des batchs de 2 tuiles. Entraînez-le sous le nom `fine_tuned_unet`, puis rangez ses scores de test dans `test_results["UNet pré-entraîné, fine-tuné"]`.

**Documentation utile**

- [Options par groupe de paramètres](https://docs.pytorch.org/docs/stable/optim.html#per-parameter-options)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `unet.encoder = unet.encoder.module.requires_grad_(True)` : retire `Frozen`, et rend les poids de l'encodeur entraînables
- `self.named_parameters()` : les paramètres du modèle, avec leur nom ; ceux de l'encodeur ont `.encoder.` dans leur nom
- `torch.optim.AdamW([{"params": encoder, "lr": 1e-5}, {"params": others, "lr": 1e-4}])` : un pas par groupe de paramètres
- `self.hparams.learning_rate` : le pas passé au constructeur d'`ImageSegmenter`
- `make_loader(df_train, 2, eval=False)` : des batchs de 2 tuiles, à passer à `train` par son argument `loader`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class FineTuningSegmenter(ImageSegmenter):
  def __init__(
    self, model: nn.Module, encoder_learning_rate: float = 1e-5, **kwargs
  ) -> None:
    super().__init__(model, **kwargs)
    self.encoder_learning_rate = encoder_learning_rate

  def configure_optimizers(self) -> torch.optim.Optimizer:
    parameters = list(self.named_parameters())
    encoder = [p for name, p in parameters if ".encoder." in name]
    others = ...
    return torch.optim.AdamW(
      [
        {"params": encoder, "lr": ...},
        {"params": others, "lr": self.hparams.learning_rate},
      ]
    )


unet.encoder = ...  # encodeur dégelé
segmenter = FineTuningSegmenter(
  pretrained_unet, encoder_learning_rate=1e-5, learning_rate=1e-4
)
fine_tuning_loader = make_loader(df_train, 2, eval=False)
trainer = train(
  segmenter, "fine_tuned_unet", epochs=fine_tuning_epochs, loader=fine_tuning_loader
)
scores = ...
test_results["UNet pré-entraîné, fine-tuné"] = scores
```

#### Solution

In [ ]:
class FineTuningSegmenter(ImageSegmenter):
  """ImageSegmenter whose encoder has its own, smaller, learning rate."""

  def __init__(
    self, model: nn.Module, encoder_learning_rate: float = 1e-5, **kwargs
  ) -> None:
    super().__init__(model, **kwargs)
    self.encoder_learning_rate = encoder_learning_rate

  def configure_optimizers(self) -> torch.optim.Optimizer:
    parameters = list(self.named_parameters())
    encoder = [p for name, p in parameters if ".encoder." in name]
    others = [p for name, p in parameters if ".encoder." not in name]
    return torch.optim.AdamW(
      [
        {"params": encoder, "lr": self.encoder_learning_rate},
        {"params": others, "lr": self.hparams.learning_rate},
      ]
    )


# Étape 2 : l'encodeur retrouve des poids entraînables et son mode train()
unet.encoder = unet.encoder.module.requires_grad_(True)
# Un pas 10 fois plus petit pour l'encodeur : ajuster ses poids sans les détruire
segmenter = FineTuningSegmenter(
  pretrained_unet, encoder_learning_rate=1e-5, learning_rate=1e-4
)

# Tout le réseau est entraîné : des batchs plus petits pour tenir en mémoire GPU
fine_tuning_loader = make_loader(df_train, 2, eval=False)
trainer = train(
  segmenter, "fine_tuned_unet", epochs=fine_tuning_epochs, loader=fine_tuning_loader
)
scores = trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")[0]
test_results["UNet pré-entraîné, fine-tuné"] = scores
print(test_results)

## Comparaison des modèles

La cellule suivante compare les scores de test des quatre modèles, ainsi que la justesse d'un modèle qui répondrait toujours la classe la plus fréquente des tuiles de test. Quel modèle obtient la meilleure IoU ? Les écarts entre les autres sont-ils assez grands pour les classer ? Pourquoi les écarts de justesse entre les modèles sont-ils plus faibles que les écarts d'IoU ?

In [ ]:
# Part des pixels de forêt dans la partie recadrée des tuiles de test
forest = shares.mean()
print(
  "Justesse en répondant toujours la classe la plus fréquente :",
  f"{max(forest, 1 - forest):.3f}",
)
pandas.DataFrame(test_results).T[["test_iou", "test_accuracy", "test_loss"]].round(3)

### Solution

Le UNet pré-entraîné et fine-tuné arrive en tête, ou à égalité en tête, à chaque exécution (environ 0,97 d'IoU) : il combine un encodeur qui a appris à reconnaître des textures sur ImageNet, les connexions de saut, qui gardent les contours précis, et un réglage de tout le réseau sur nos images infrarouges. Les trois autres modèles sont à un à trois points d'IoU les uns des autres, et leur ordre change d'une exécution à l'autre : sur cette tâche facile (forêt ou non-forêt), ces écarts sont de l'ordre de la variation d'un entraînement à l'autre. Ils ne permettent donc pas de dire si les connexions de saut, le pré-entraînement ou la quantité de données font la différence. Pour les départager, il faudrait répéter chaque entraînement avec plusieurs graines et comparer les moyennes, ou regarder les tuiles à lisières affichées plus haut, où les erreurs se voient.

La justesse compte tous les pixels, dont la grande majorité est facile : l'intérieur des zones de forêt ou de non-forêt. Elle se lit mal sans la justesse du modèle qui répond toujours la classe la plus fréquente : tous les modèles la dépassent largement, ce qui montre seulement qu'ils ont appris quelque chose, et leurs valeurs, serrées près de 1, cachent comment ils se trompent. L'IoU ne compte que les pixels que la prédiction ou la cible disent de la forêt : chaque pixel de forêt manqué ou en trop y pèse davantage, et elle étale donc un peu plus les écarts que la justesse. Elle les rend plus lisibles, sans les rendre significatifs pour autant.